# SuspiciousLogin Dataset — ML Benchmark

Reproduces the results in `docs/BENCHMARK_RESULTS_EN.md`: Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, and Isolation Forest, compared against the empirical `risk_score` (v3), on a chronological 70/15/15 split -- plus confusion matrices, ROC/PR/calibration curves, feature importance, a cold-start/warm-start/mature-profile breakdown, and a feature-group ablation study.

**Privacy notice**: this notebook needs the **restricted** file (`suspicious_logins_restricted_v1.csv`, with `event_time`), not the public one. If running this on Kaggle, you are uploading pseudonymized but still sensitive data to a third-party platform -- make sure the dataset you upload is set to **private**, never public.

**Full methodology**: see `docs/RISK_SCORE_METHODOLOGY_EN.md` and `docs/BENCHMARK_RESULTS_EN.md` in the repository.

## 1. Environment

In [ ]:
# On Kaggle, xgboost and lightgbm are usually already installed -- this
# only installs if actually missing, without forcing an unnecessary reinstall.
import importlib
for package in ["xgboost", "lightgbm"]:
    if importlib.util.find_spec(package) is None:
        !pip install {package} --quiet
print("environment OK")

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay,
                             RocCurveDisplay, average_precision_score,
                             brier_score_loss, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

print(f"XGBoost available: {HAS_XGBOOST}")
print(f"LightGBM available: {HAS_LIGHTGBM}")

## 2. Data path

**Adjust `RESTRICTED_FILE` below** to wherever you uploaded the dataset on Kaggle -- typically something like `/kaggle/input/<your-dataset-name>/suspicious_logins_restricted_v1.csv`. Outside Kaggle (local execution), the fallback path already works unchanged if the private repository's folder structure is kept.

In [ ]:
# ADJUST THIS PATH when uploading to Kaggle
RESTRICTED_FILE = Path("/kaggle/input/suspiciouslogin-restricted/suspicious_logins_restricted_v1.csv")

# fallback for local execution outside Kaggle
if not RESTRICTED_FILE.exists():
    RESTRICTED_FILE = Path("data/processed/suspicious_logins_restricted_v1.csv")

if not RESTRICTED_FILE.exists():
    raise FileNotFoundError(
        f"Restricted file not found at any of the tried paths. Confirm the "
        f"exact path of the dataset you uploaded on Kaggle, and update "
        f"RESTRICTED_FILE above."
    )
print(f"using: {RESTRICTED_FILE}")

## 3. Configuration — identical to `benchmark_ml.py`

In [ ]:
# Confirmed with the (first) institution: June-August is the Angolan
# academic vacation period. Event volume drops sharply (~6,500-8,000/month
# to 1,440-3,172/month) and the suspicious rate drops even more sharply
# (~7-9% to ~1.3-1.5%) starting specifically in July -- confirmed NOT
# caused by a security policy change in this period. Excluded from the
# train/val/test split (kept in the published dataset itself). See
# docs/RISK_SCORE_METHODOLOGY_EN.md for the full justification.
EVALUATION_CUTOFF_DATE = "2026-07-01"

# Features actually fed to the models. Deliberately excludes:
#   actor_pseudo_id -- identifier, not a predictive feature
#   risk_score, risk_level -- the comparison baseline, not an input
#   label -- the target
#   event_time -- used only for the split, not as a feature
NUMERIC_FEATURES = [
    "event_hour", "day_of_week", "month", "quarter",
    "is_weekend", "is_business_hours", "is_night_login",
    "ip_version", "new_ip", "distinct_ips_7d", "distinct_ips_30d",
    "is_new_country", "is_new_region", "is_new_city",
    "distinct_countries_cumulative", "distinct_regions_cumulative",
    "distinct_cities_cumulative", "country_changed", "continent_changed",
    "countries_seen_30d", "regions_seen_30d", "cities_seen_30d",
    "logins_24h", "logins_7d", "logins_30d", "avg_logins_per_day",
    "hours_since_last_login", "days_since_first_login",
    "abnormal_login_hour", "abnormal_weekday",
    "impossible_travel", "travel_distance_km", "travel_speed_kmh",
    "multiple_country_logins_24h", "distinct_users_per_network_24h",
    "geo_jump", "history_available",
]
CATEGORICAL_FEATURES = ["ip_country", "continent", "login_type"]
TOP_K_CATEGORIES = 10  # per categorical column; rarer values -> "_other_"

# Feature groups for the ablation study (section 12) -- matches the
# incremental M1..M4 structure: time+auth, +geography, +user history,
# +network. One column can only belong to one group.
FEATURE_GROUPS = {
    "M1_time_auth": [
        "event_hour", "day_of_week", "month", "quarter", "is_weekend",
        "is_business_hours", "is_night_login",
    ],
    "M2_geography": [
        "is_new_country", "is_new_region", "is_new_city",
        "distinct_countries_cumulative", "distinct_regions_cumulative",
        "distinct_cities_cumulative", "country_changed", "continent_changed",
        "countries_seen_30d", "regions_seen_30d", "cities_seen_30d",
        "geo_jump", "multiple_country_logins_24h",
    ],
    "M3_user_history": [
        "logins_24h", "logins_7d", "logins_30d", "avg_logins_per_day",
        "hours_since_last_login", "days_since_first_login",
        "abnormal_login_hour", "abnormal_weekday", "history_available",
    ],
    "M4_network": [
        "new_ip", "distinct_ips_7d", "distinct_ips_30d",
        "distinct_users_per_network_24h", "impossible_travel",
        "travel_distance_km", "travel_speed_kmh", "ip_version",
    ],
}
print(f"{len(NUMERIC_FEATURES)} numeric features, {len(CATEGORICAL_FEATURES)} categorical")

## 4. Helper functions — identical to `src/benchmark_ml.py`

In [ ]:
def engineer_challenge_method_flags(df):
    """login_challenge_method -> a handful of interpretable binary flags,
    instead of one-hot encoding every distinct pipe-separated combination
    (which would blow up dimensionality on a genuinely high-cardinality,
    largely uninformative-per-combination field)."""
    methods = df["login_challenge_method"].fillna("").astype(str)
    out = pd.DataFrame(index=df.index)
    for flag_name, token in [
        ("uses_password", "password"),
        ("uses_passkey", "passkey"),
        ("uses_device_prompt", "device_prompt"),
        ("uses_preregistered_phone", "idv_preregistered_phone"),
        ("uses_preregistered_email", "idv_preregistered_email"),
    ]:
        out[flag_name] = methods.str.contains(token, regex=False).astype(int)
    out["num_challenge_methods"] = methods.apply(
        lambda v: len(set(v.split("|"))) if v else 0)
    return out

In [ ]:
def fit_category_encoder(train_series, top_k):
    """Returns the set of categories to keep as their own one-hot column
    (the top_k most frequent IN TRAIN ONLY); anything else, in any split,
    maps to '_other_'. Fitting on train only is what prevents a category
    that's only common in val/test from leaking distributional information
    back into the encoding."""
    return set(train_series.value_counts().head(top_k).index)


def apply_category_encoder(series, keep_categories, prefix):
    capped = series.where(series.isin(keep_categories), other="_other_")
    return pd.get_dummies(capped, prefix=prefix, dtype=int)

In [ ]:
def build_features(df, keep_categories_by_col, numeric_features=None):
    """numeric_features defaults to the full NUMERIC_FEATURES list; the
    ablation study (section 12) passes a restricted subset instead, to
    isolate the contribution of each feature group."""
    if numeric_features is None:
        numeric_features = NUMERIC_FEATURES
    df = df.copy()
    # history_available: whether this user has any PRIOR event at all,
    # derived from hours_since_last_login (0 specifically and only on a
    # user's first-ever observed event). Confirmed genuinely predictive on
    # its own: real suspicious rate in this dataset is 2.69% among users
    # with prior history vs 7.69% on their first observed event.
    df["history_available"] = (df["hours_since_last_login"] > 0).astype(int)

    numeric = df[[c for c in numeric_features if c in df.columns]].copy()
    # A small number of rows have missing geo data at the source (see
    # docs/DATA_AUDIT_REPORT_EN.md), which cascades into NaN in every
    # derived feature depending on ip_region/ip_country/etc. Filled with 0:
    # for count-like features this means "no known distinct value counted,"
    # the correct behavior for a row with unknown geo, not a guess.
    numeric = numeric.fillna(0)
    challenge_flags = engineer_challenge_method_flags(df)
    cat_frames = [
        apply_category_encoder(df[col], keep_categories_by_col[col], col)
        for col in CATEGORICAL_FEATURES
    ]
    return pd.concat([numeric, challenge_flags] + cat_frames, axis=1)

In [ ]:
def cap_dominant_users_in_training(train_df, max_events_per_user=80, random_state=42):
    """Subsamples any user's events DOWN to max_events_per_user, applied
    ONLY to the training split -- validation and test must reflect the
    real, honest distribution the model will actually be judged against.

    max_events_per_user=80 is the 99th percentile of events-per-user
    within the training split at the time it was chosen -- a deliberately
    surgical cut affecting only the most extreme outliers."""
    counts = train_df["actor_pseudo_id"].value_counts()
    over_cap = counts[counts > max_events_per_user]
    if len(over_cap) == 0:
        print(f"No user exceeds {max_events_per_user} events in training.")
        return train_df

    keep_indices = []
    rng = np.random.RandomState(random_state)
    for user, group in train_df.groupby("actor_pseudo_id"):
        if len(group) > max_events_per_user:
            keep_indices.extend(
                rng.choice(group.index, size=max_events_per_user, replace=False))
        else:
            keep_indices.extend(group.index)

    capped = train_df.loc[sorted(keep_indices)].reset_index(drop=True)
    n_removed = len(train_df) - len(capped)
    print(f"Capped {len(over_cap)} user(s) exceeding {max_events_per_user} events "
         f"in training -- removed {n_removed} rows ({n_removed/len(train_df)*100:.2f}% "
         f"of training), validation/test untouched")
    top_user_pct_before = counts.max() / len(train_df) * 100
    top_user_pct_after = min(counts.max(), max_events_per_user) / len(capped) * 100
    print(f"  most active user's share of training: "
         f"{top_user_pct_before:.2f}% -> {top_user_pct_after:.2f}%")
    return capped

In [ ]:
def align_columns(train_X, other_X):
    """Ensures val/test have exactly the same columns as train."""
    return other_X.reindex(columns=train_X.columns, fill_value=0)


def false_positives_per_1000(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return (fp / len(y_true)) * 1000


def select_threshold_on_validation(y_val, val_scores):
    """Chooses the score threshold that maximizes F1 on the VALIDATION
    split, never the test split."""
    candidates = np.unique(val_scores)
    best_threshold, best_f1 = 0.5, -1.0
    for t in candidates:
        y_pred = (val_scores >= t).astype(int)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_threshold = f1, t
    return best_threshold


def evaluate(name, y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "model": name,
        "threshold_used": threshold,
        "AUC-ROC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, scores),
        "FP_per_1000_logins": false_positives_per_1000(y_true, y_pred),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    }

## 5. Load data, exclude vacation period, split into train/validation/test

In [ ]:
df = pd.read_csv(RESTRICTED_FILE)
df["event_time"] = pd.to_datetime(df["event_time"], format="ISO8601")
df = df.sort_values("event_time").reset_index(drop=True)

n_before_cutoff = len(df)
df = df[df["event_time"] < EVALUATION_CUTOFF_DATE].reset_index(drop=True)
print(f"Excluded {n_before_cutoff - len(df)} row(s) on/after {EVALUATION_CUTOFF_DATE} "
     f"(academic vacation period, see docs/RISK_SCORE_METHODOLOGY_EN.md) from "
     f"the train/val/test split -- still present in the published dataset itself")

In [ ]:
relevant_columns = ([c for c in NUMERIC_FEATURES if c != "history_available"]
                   + CATEGORICAL_FEATURES
                   + ["login_challenge_method", "label", "event_time"])
n_before = len(df)
rows_with_gaps = df[relevant_columns].isna().any(axis=1)
n_dropped = int(rows_with_gaps.sum())
if n_dropped:
    rate_dropped = df.loc[rows_with_gaps, "label"].mean()
    rate_kept = df.loc[~rows_with_gaps, "label"].mean()
    print(f"Dropping {n_dropped} of {n_before} rows with a missing value "
         f"in a used column ({n_dropped/n_before*100:.2f}%)")
    print(f"  positive rate in dropped rows: {rate_dropped*100:.2f}%  "
         f"vs kept rows: {rate_kept*100:.2f}%")
df = df.loc[~rows_with_gaps].reset_index(drop=True)

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

t_tr_min, t_tr_max = train_df['event_time'].min(), train_df['event_time'].max()
t_va_min, t_va_max = val_df['event_time'].min(), val_df['event_time'].max()
t_te_min, t_te_max = test_df['event_time'].min(), test_df['event_time'].max()
print(f"train: {len(train_df)} ({t_tr_min} to {t_tr_max})")
print(f"val:   {len(val_df)} ({t_va_min} to {t_va_max})")
print(f"test:  {len(test_df)} ({t_te_min} to {t_te_max})")
print(f"positive rate -- train: {train_df['label'].mean()*100:.2f}%  "
     f"val: {val_df['label'].mean()*100:.2f}%  test: {test_df['label'].mean()*100:.2f}%")

In [ ]:
train_df = cap_dominant_users_in_training(train_df, max_events_per_user=80)

## 6. Build features

In [ ]:
keep_categories_by_col = {
    col: fit_category_encoder(train_df[col], TOP_K_CATEGORIES)
    for col in CATEGORICAL_FEATURES
}

X_train = build_features(train_df, keep_categories_by_col)
X_val = align_columns(X_train, build_features(val_df, keep_categories_by_col))
X_test = align_columns(X_train, build_features(test_df, keep_categories_by_col))
y_train, y_val, y_test = train_df["label"], val_df["label"], test_df["label"]

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

## 7. Train and evaluate the supervised models

In [ ]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=8, random_state=42),
    "RandomForest": RandomForestClassifier(
        class_weight="balanced", n_estimators=200, max_depth=10,
        random_state=42, n_jobs=-1),
}
if HAS_XGBOOST:
    models["XGBoost"] = XGBClassifier(
        scale_pos_weight=pos_weight, n_estimators=200, max_depth=6,
        eval_metric="logloss", random_state=42)
if HAS_LIGHTGBM:
    models["LightGBM"] = LGBMClassifier(
        class_weight="balanced", n_estimators=200, max_depth=6,
        random_state=42, verbosity=-1)

fitted_models = {}
results = []
test_scores_by_model = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_scores = model.predict_proba(X_val)[:, 1]
    test_scores = model.predict_proba(X_test)[:, 1]
    threshold = select_threshold_on_validation(y_val, val_scores)
    results.append(evaluate(name, y_test, test_scores, threshold))
    fitted_models[name] = model
    test_scores_by_model[name] = (test_scores, threshold)
    print(f"{name}: trained")

## 8. Isolation Forest (unsupervised) and the `risk_score` baseline

In [ ]:
iso = IsolationForest(
    n_estimators=200, contamination=max(y_train.mean(), 0.01), random_state=42)
iso.fit(X_train)
iso_scores_val_raw = -iso.score_samples(X_val)
iso_scores_test_raw = -iso.score_samples(X_test)
iso_min, iso_max = iso_scores_val_raw.min(), iso_scores_val_raw.max()
iso_scores_val = np.clip((iso_scores_val_raw - iso_min) / (iso_max - iso_min + 1e-9), 0, 1)
iso_scores_test = np.clip((iso_scores_test_raw - iso_min) / (iso_max - iso_min + 1e-9), 0, 1)
iso_threshold = select_threshold_on_validation(y_val, iso_scores_val)
results.append(evaluate("IsolationForest", y_test, iso_scores_test, iso_threshold))
test_scores_by_model["IsolationForest"] = (iso_scores_test, iso_threshold)
print("Isolation Forest: evaluated")

In [ ]:
risk_val_raw = val_df["risk_score"].to_numpy()
risk_test_raw = test_df["risk_score"].to_numpy()
risk_min, risk_max = risk_val_raw.min(), risk_val_raw.max()
risk_scores_val = np.clip((risk_val_raw - risk_min) / (risk_max - risk_min + 1e-9), 0, 1)
risk_scores_test = np.clip((risk_test_raw - risk_min) / (risk_max - risk_min + 1e-9), 0, 1)
risk_threshold = select_threshold_on_validation(y_val, risk_scores_val)
results.append(evaluate("risk_score (baseline)", y_test, risk_scores_test, risk_threshold))
test_scores_by_model["risk_score (baseline)"] = (risk_scores_test, risk_threshold)
print("risk_score: evaluated")

## 9. Results table

In [ ]:
results_df = pd.DataFrame(results).set_index("model")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
results_df.round(4)

In [ ]:
results_df.to_csv("benchmark_results.csv")
print("Saved benchmark_results.csv")
print()
print("Compare this table with docs/BENCHMARK_RESULTS_EN.md -- the numbers should")
print("match exactly, since the data, split, and code are the same.")

## 10. Confusion matrices

One panel per model, using each model's own validation-selected threshold (not a fixed 0.5).

In [ ]:
model_names = list(test_scores_by_model.keys())
n_models = len(model_names)
n_cols = 3
n_rows = -(-n_models // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = axes.flatten()

for i, name in enumerate(model_names):
    scores, threshold = test_scores_by_model[name]
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["normal", "suspicious"])
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(f"{name}\n(threshold={threshold:.3f})", fontsize=10)

for j in range(len(model_names), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()

## 11. ROC, Precision-Recall, and calibration curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name in model_names:
    scores, _ = test_scores_by_model[name]
    RocCurveDisplay.from_predictions(y_test, scores, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_title("ROC curves — test set")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
base_rate = y_test.mean()
for name in model_names:
    scores, _ = test_scores_by_model[name]
    PrecisionRecallDisplay.from_predictions(y_test, scores, name=name, ax=ax)
ax.axhline(base_rate, linestyle="--", color="gray", label=f"Base rate ({base_rate:.3f})")
ax.set_title("Precision-Recall curves — test set")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.savefig("pr_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name in model_names:
    scores, _ = test_scores_by_model[name]
    frac_pos, mean_pred = calibration_curve(y_test, scores, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name, markersize=4)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.set_xlabel("Mean predicted score (per bin)")
ax.set_ylabel("Observed positive fraction (per bin)")
ax.set_title("Calibration curves — test set (10 quantile bins)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("calibration_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 12. Feature importance

Tree-based models only (Logistic Regression's coefficients aren't directly comparable in scale without additional standardization, and Isolation Forest/`risk_score` aren't trained on the full feature set the same way).

In [ ]:
tree_models = {n: m for n, m in fitted_models.items()
              if n in ("DecisionTree", "RandomForest", "XGBoost", "LightGBM")}
fig, axes = plt.subplots(1, len(tree_models), figsize=(6 * len(tree_models), 6))
if len(tree_models) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, tree_models.items()):
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    top15 = importances.sort_values(ascending=True).tail(15)
    top15.plot(kind="barh", ax=ax)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

## 13. Cold-start / warm-start / mature-profile breakdown

Directly ties to a central characteristic of this dataset: 96.3% of users have only a single observed event overall (see `docs/DATA_AUDIT_REPORT_EN.md`). This breaks the test set into three groups by how much prior history each user actually had *at the time of that specific event* (not their final total, to avoid leakage):

- **Cold-start**: no prior event at all (`prior_events_total = 0`)
- **Warm-start**: at least 1 prior event, fewer than 10
- **Mature profile**: at least 10 prior events

`prior_events_total` is computed here only for this breakdown (not used as a model input) via a per-user running count on the already time-sorted data.

In [ ]:
df_sorted = df.sort_values("event_time").reset_index(drop=True)
df_sorted["prior_events_total"] = df_sorted.groupby("actor_pseudo_id").cumcount()

test_index = test_df.index
prior_events_test = df_sorted.loc[test_index, "prior_events_total"]

def profile_bucket(n):
    if n == 0:
        return "cold-start"
    elif n < 10:
        return "warm-start"
    return "mature profile"

profile_test = prior_events_test.apply(profile_bucket)
print(profile_test.value_counts())
print()
print("Positive rate by profile:")
print(pd.DataFrame({"label": y_test.values, "profile": profile_test.values})
     .groupby("profile")["label"].mean().round(4))

In [ ]:
best_model_name = results_df["AUC-ROC"].idxmax()
best_scores, best_threshold = test_scores_by_model[best_model_name]
print(f"Breaking down the best model ({best_model_name}) by profile:\n")

profile_rows = []
for profile in ["cold-start", "warm-start", "mature profile"]:
    mask = (profile_test == profile).values
    if mask.sum() < 10:
        continue
    y_sub = y_test.values[mask]
    scores_sub = best_scores[mask]
    y_pred_sub = (scores_sub >= best_threshold).astype(int)
    row = {
        "profile": profile,
        "n": int(mask.sum()),
        "positive_rate": y_sub.mean(),
        "AUC-ROC": roc_auc_score(y_sub, scores_sub) if len(set(y_sub)) > 1 else float("nan"),
        "precision": precision_score(y_sub, y_pred_sub, zero_division=0),
        "recall": recall_score(y_sub, y_pred_sub, zero_division=0),
        "F1": f1_score(y_sub, y_pred_sub, zero_division=0),
    }
    profile_rows.append(row)

profile_results = pd.DataFrame(profile_rows).set_index("profile")
profile_results.round(4)

## 14. Ablation study — feature-group contribution

Trains a Random Forest (fixed algorithm, to isolate the effect of features rather than algorithm choice) on incrementally larger feature groups: M1 (time + authentication) → M2 (+ geography) → M3 (+ user history) → M4 (+ network), compared against M5 (the `risk_score` baseline) and M6 (the best full model from section 9).

Directly addresses: *how much does behavioral history improve prediction, and for what share of users is that gain actually available?* -- given 96.3% of users are single-event, this is a central question for the dataset.

In [ ]:
ablation_results = []
cumulative_numeric = []
group_order = ["M1_time_auth", "M2_geography", "M3_user_history", "M4_network"]

for step_name in group_order:
    cumulative_numeric += FEATURE_GROUPS[step_name]
    X_train_step = build_features(train_df, keep_categories_by_col, numeric_features=cumulative_numeric)
    X_val_step = align_columns(X_train_step, build_features(val_df, keep_categories_by_col, numeric_features=cumulative_numeric))
    X_test_step = align_columns(X_train_step, build_features(test_df, keep_categories_by_col, numeric_features=cumulative_numeric))

    rf_step = RandomForestClassifier(
        class_weight="balanced", n_estimators=200, max_depth=10,
        random_state=42, n_jobs=-1)
    rf_step.fit(X_train_step, y_train)
    val_scores_step = rf_step.predict_proba(X_val_step)[:, 1]
    test_scores_step = rf_step.predict_proba(X_test_step)[:, 1]
    threshold_step = select_threshold_on_validation(y_val, val_scores_step)
    row = evaluate(step_name, y_test, test_scores_step, threshold_step)
    row["n_features"] = X_train_step.shape[1]
    ablation_results.append(row)
    print(f"{step_name}: done ({X_train_step.shape[1]} features)")

# M5: risk_score baseline (already computed in section 8)
row_m5 = evaluate("M5_risk_score", y_test, risk_scores_test, risk_threshold)
row_m5["n_features"] = 6
ablation_results.append(row_m5)

# M6: best full model from section 9
row_m6 = evaluate("M6_best_full_model", y_test, best_scores, best_threshold)
row_m6["n_features"] = X_train.shape[1]
ablation_results.append(row_m6)

ablation_df = pd.DataFrame(ablation_results).set_index("model")
ablation_df[["n_features", "AUC-ROC", "PR-AUC", "precision", "recall", "F1"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_order = ["M1_time_auth", "M2_geography", "M3_user_history", "M4_network",
             "M5_risk_score", "M6_best_full_model"]
plot_df = ablation_df.loc[[m for m in plot_order if m in ablation_df.index]]
ax.plot(plot_df.index, plot_df["AUC-ROC"], marker="o", label="AUC-ROC")
ax.plot(plot_df.index, plot_df["PR-AUC"], marker="s", label="PR-AUC")
ax.set_xticklabels(plot_df.index, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Ablation study: incremental feature-group contribution")
ax.legend()
plt.tight_layout()
plt.savefig("ablation_study.png", dpi=120, bbox_inches="tight")
plt.show()

## 15. Summary

This notebook reproduces the main benchmark table exactly, and adds:
confusion matrices, ROC/PR/calibration curves, tree-model feature
importance, a cold-start/warm-start/mature-profile breakdown, and a
feature-group ablation study. All figures are saved as `.png` files in
the working directory alongside `benchmark_results.csv`.

For dataset-level statistics (coverage, Gini index, Lorenz curve, data
quality, missing values) and empirical pattern analysis (temporal/
geographic risk ratios, statistical tests), see the separate planned
notebooks for Article 1 and Article 2 -- out of scope for this
ML-benchmark notebook.